In [2]:
import os

path = r"C:\Users\Admin\Downloads\infosys\data\raw\patients.csv"
print(os.path.exists(path))

True


In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\patients.csv")
df

,Patient_ID,Name,Gender,Age,Blood_Group,City
0,P00001,Patient_1,Male,59,A+,Tiruppur
1,P00002,Patient_2,Female,65,B+,Trichy
2,P00003,Patient_3,Male,81,O+,Tiruppur
3,P00004,Patient_4,Male,20,O-,Coimbatore
4,P00005,Patient_5,Male,77,O-,Salem
...,...,...,...,...,...,...
4995,P04996,Patient_4996,Male,48,O-,Coimbatore
4996,P04997,Patient_4997,Male,47,AB+,Chennai
4997,P04998,Patient_4998,Male,8,AB+,Salem
4998,P04999,Patient_4999,Male,83,A+,Salem


In [4]:
print(f"info: {df.info}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


info: <bound method DataFrame.info of      Patient_ID          Name  Gender  Age Blood_Group        City
0        P00001     Patient_1    Male   59          A+    Tiruppur
1        P00002     Patient_2  Female   65          B+      Trichy
2        P00003     Patient_3    Male   81          O+    Tiruppur
3        P00004     Patient_4    Male   20          O-  Coimbatore
4        P00005     Patient_5    Male   77          O-       Salem
...         ...           ...     ...  ...         ...         ...
4995     P04996  Patient_4996    Male   48          O-  Coimbatore
4996     P04997  Patient_4997    Male   47         AB+     Chennai
4997     P04998  Patient_4998    Male    8         AB+       Salem
4998     P04999  Patient_4999    Male   83          A+       Salem
4999     P05000  Patient_5000  Female   71          A+     Chennai

[5000 rows x 6 columns]>
Shape: (5000, 6)
Columns: ['Patient_ID', 'Name', 'Gender', 'Age', 'Blood_Group', 'City']


In [5]:
print(df.head())

  Patient_ID       Name  Gender  Age Blood_Group        City
0     P00001  Patient_1    Male   59          A+    Tiruppur
1     P00002  Patient_2  Female   65          B+      Trichy
2     P00003  Patient_3    Male   81          O+    Tiruppur
3     P00004  Patient_4    Male   20          O-  Coimbatore
4     P00005  Patient_5    Male   77          O-       Salem


In [6]:
print(f"\nDtypes:\n{df.dtypes}")

print(f"\nNull counts:\n{df.isna().sum()}")


Dtypes:
Patient_ID     object
Name           object
Gender         object
Age             int64
Blood_Group    object
City           object
dtype: object

Null counts:
Patient_ID     0
Name           0
Gender         0
Age            0
Blood_Group    0
City           0
dtype: int64


In [7]:
print(f"\nDuplicate rows: {df.duplicated().sum()}")

for col in df.select_dtypes(include='object').columns:
    print(f"\nUnique values in '{col}': {sorted(df[col].astype(str).unique())[:20]}")


Duplicate rows: 0

Unique values in 'Patient_ID': ['P00001', 'P00002', 'P00003', 'P00004', 'P00005', 'P00006', 'P00007', 'P00008', 'P00009', 'P00010', 'P00011', 'P00012', 'P00013', 'P00014', 'P00015', 'P00016', 'P00017', 'P00018', 'P00019', 'P00020']

Unique values in 'Name': ['Patient_1', 'Patient_10', 'Patient_100', 'Patient_1000', 'Patient_1001', 'Patient_1002', 'Patient_1003', 'Patient_1004', 'Patient_1005', 'Patient_1006', 'Patient_1007', 'Patient_1008', 'Patient_1009', 'Patient_101', 'Patient_1010', 'Patient_1011', 'Patient_1012', 'Patient_1013', 'Patient_1014', 'Patient_1015']

Unique values in 'Gender': ['Female', 'Male']

Unique values in 'Blood_Group': ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']

Unique values in 'City': ['Chennai', 'Coimbatore', 'Madurai', 'Salem', 'Tiruppur', 'Trichy']


In [8]:
df.nunique()

Patient_ID     5000
Name           5000
Gender            2
Age              90
Blood_Group       8
City              6
dtype: int64

In [9]:
print("STANDARDIZE GENDER")

GENDER_MAP = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female', 'femal': 'Female',
    'o': 'Other', 'other': 'Other', 'others': 'Other'
}

before = sorted(df['Gender'].astype(str).unique())
df['Gender'] = (
    df['Gender'].astype(str).str.strip().str.lower()
      .map(GENDER_MAP)
      .fillna(df['Gender'].astype(str).str.strip().str.title())
)
after = sorted(df['Gender'].unique())

print(f"Before: {before}")
print(f"After : {after}")
assert set(df['Gender'].unique()) <= {'Male', 'Female', 'Other'}, \
    "Gender contains values outside Male/Female/Other"
print("PASSED: Gender restricted to Male / Female / Other")

STANDARDIZE GENDER
Before: ['Female', 'Male']
After : ['Female', 'Male']
PASSED: Gender restricted to Male / Female / Other


In [10]:
print(" VALIDATE DATA TYPES & RANGES")

# Age must be numeric and within a plausible human range
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
invalid_age = df['Age'].isna().sum()
if invalid_age > 0:
    print(f"WARNING: {invalid_age} non-numeric Age value(s) coerced to NaN")

print(f"Age range: {df['Age'].min()} - {df['Age'].max()}")

# Blood_Group restricted to known valid set
VALID_BLOOD_GROUPS = {'A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-'}
df['Blood_Group'] = df['Blood_Group'].astype(str).str.strip().str.upper()
invalid_bg = df[~df['Blood_Group'].isin(VALID_BLOOD_GROUPS)]
print(f"Invalid Blood_Group values: {invalid_bg['Blood_Group'].unique().tolist()}")

# City -> trim + title case (consistency only, no invention of new values)
df['City'] = df['City'].astype(str).str.strip().str.title()
print(f"City values: {sorted(df['City'].unique())}")


 VALIDATE DATA TYPES & RANGES
Age range: 1 - 90
Invalid Blood_Group values: []
City values: ['Chennai', 'Coimbatore', 'Madurai', 'Salem', 'Tiruppur', 'Trichy']


In [11]:
print("FLAG CLINICAL ANOMALIES")
 
anomaly_mask = (df['Age'] > 110) | (df['Age'] < 0) | (df['Age'].isna())
if 'Blood_Pressure' in df.columns:
    anomaly_mask = anomaly_mask | (df['Blood_Pressure'] == 0)
 
df['Anomaly_Flag'] = anomaly_mask
print(f"Rows flagged as anomalies: {int(anomaly_mask.sum())}")
print(df['Anomaly_Flag'].value_counts())


FLAG CLINICAL ANOMALIES
Rows flagged as anomalies: 0
Anomaly_Flag
False    5000
Name: count, dtype: int64


In [14]:
admissions = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\admissions.csv")
print(admissions.head())



  Admission_ID Patient_ID Doctor_ID Department_ID Admission_Date  \
0       A00001     P00001   DR01858          D012     2026-01-20   
1       A00002     P00002   DR03993          D015     2026-02-04   
2       A00003     P00003   DR03106          D004     2025-09-22   
3       A00004     P00004   DR04234          D007     2025-05-26   
4       A00005     P00005   DR04566          D012     2026-03-13   

  Discharge_Date  Bed_No Admission_Type  Length_of_Stay           Status  
0     2026-01-23      23      Emergency               3        Recovered  
1     2026-02-12     460        Walk-in               8  Under Treatment  
2     2025-09-24      33        Walk-in               2       Discharged  
3     2025-06-05     485      Emergency              10        Recovered  
4     2026-03-16     308      Emergency               3        Recovered  


In [15]:
import pandas as pd

patients = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\patients.csv")
admissions = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\admissions.csv")

# Check if all Patient_IDs in admissions exist in patients
missing_ids = admissions[~admissions["Patient_ID"].isin(patients["Patient_ID"])]

if missing_ids.empty:
    print("All Patient_IDs in admissions.csv exist in patients.csv")
else:
    print("Some Patient_IDs are missing in patients.csv")
    print("Count of missing IDs:", missing_ids["Patient_ID"].nunique())
    print("Example missing IDs:", missing_ids["Patient_ID"].unique()[:10])


All Patient_IDs in admissions.csv exist in patients.csv


In [ ]:
print("FINAL VALIDATION & SAVE")

checks = {
    "No nulls remaining":            df.isna().sum().sum() == 0,
    "No duplicate rows":             df.duplicated().sum() == 0,
    "Patient_ID is unique":          df['Patient_ID'].is_unique,
    "Gender in {Male,Female,Other}": set(df['Gender'].unique()) <= {'Male', 'Female', 'Other'},
   # "'Name' column removed":         'Name' not in df.columns,
}

for check, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

assert all(checks.values()), "Validation failed — see checks above."
print(f"\nFinal shape: {df.shape}")
print(df.head(5).to_string(index=False))



In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned file to: {C:\Users\Admin\Downloads\infosys}")

In [16]:
df.to_csv("patient_clean.csv", index=False)
df

,Patient_ID,Name,Gender,Age,Blood_Group,City,Anomaly_Flag
0,P00001,Patient_1,Male,59,A+,Tiruppur,False
1,P00002,Patient_2,Female,65,B+,Trichy,False
2,P00003,Patient_3,Male,81,O+,Tiruppur,False
3,P00004,Patient_4,Male,20,O-,Coimbatore,False
4,P00005,Patient_5,Male,77,O-,Salem,False
...,...,...,...,...,...,...,...
4995,P04996,Patient_4996,Male,48,O-,Coimbatore,False
4996,P04997,Patient_4997,Male,47,AB+,Chennai,False
4997,P04998,Patient_4998,Male,8,AB+,Salem,False
4998,P04999,Patient_4999,Male,83,A+,Salem,False
